In [3]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import antigravity
from pathlib import Path
import json

PROJECT_ROOT = Path("/content/drive/MyDrive/image-captioning-rag")

RAW_ROOT = PROJECT_ROOT / "00_original_data"

TRAIN_IMAGES = RAW_ROOT / "train2017"
VAL_IMAGES = RAW_ROOT / "val2017"
ANNOTATIONS = RAW_ROOT / "annotations"

TRAIN_CAPTIONS = ANNOTATIONS / "captions_train2017.json"
VAL_CAPTIONS = ANNOTATIONS / "captions_val2017.json"

print("Training images:", len(list(TRAIN_IMAGES.glob("*.jpg"))))
print("Validation images:", len(list(VAL_IMAGES.glob("*.jpg"))))

print("Training annotations exist:", TRAIN_CAPTIONS.exists())
print("Validation annotations exist:", VAL_CAPTIONS.exists())

Training images: 0
Validation images: 5000
Training annotations exist: True
Validation annotations exist: True


In [5]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/image-captioning-rag")

print("Project exists:", PROJECT_ROOT.exists())
print("Project path:", PROJECT_ROOT)

Project exists: True
Project path: /content/drive/MyDrive/image-captioning-rag


In [6]:
for item in PROJECT_ROOT.iterdir():
    print(item.name)

00_original_data
01_manifests
03_embeddings
04_indexes
06_releases
reports
01_splits
02_processed_data
artifacts


In [7]:
RAW_ROOT = PROJECT_ROOT / "00_original_data"

print("Raw data exists:", RAW_ROOT.exists())

if RAW_ROOT.exists():
    for item in RAW_ROOT.iterdir():
        print(item.name, "| Directory:", item.is_dir())

Raw data exists: True
train2017 | Directory: True
annotations | Directory: True
val2017 | Directory: True


In [8]:
TRAIN_IMAGES = RAW_ROOT / "train2017"
VAL_IMAGES = RAW_ROOT / "val2017"
ANNOTATIONS = RAW_ROOT / "annotations"

TRAIN_CAPTIONS = ANNOTATIONS / "captions_train2017.json"
VAL_CAPTIONS = ANNOTATIONS / "captions_val2017.json"

print("Training folder:", TRAIN_IMAGES.exists())
print("Validation folder:", VAL_IMAGES.exists())
print("Annotations folder:", ANNOTATIONS.exists())

print("Training annotations:", TRAIN_CAPTIONS.exists())
print("Validation annotations:", VAL_CAPTIONS.exists())

Training folder: True
Validation folder: True
Annotations folder: True
Training annotations: True
Validation annotations: True


In [9]:
print("Training images:", len(list(TRAIN_IMAGES.glob("*.jpg"))))
print("Validation images:", len(list(VAL_IMAGES.glob("*.jpg"))))

Training images: 61705
Validation images: 5000


In [10]:
from pathlib import Path

TRAIN_IMAGES = PROJECT_ROOT / "00_original_data" / "train2017"

print("Folder exists:", TRAIN_IMAGES.exists())
print("Is directory:", TRAIN_IMAGES.is_dir())

files = list(TRAIN_IMAGES.iterdir())

print("Total entries:", len(files))
print("\nFirst 15 entries:")

for file in files[:15]:
    print(file.name, "|", file.suffix, "|", file.is_file())

Folder exists: True
Is directory: True
Total entries: 61705

First 15 entries:
000000342159.jpg | .jpg | True
000000549996.jpg | .jpg | True
000000145767.jpg | .jpg | True
000000383764.jpg | .jpg | True
000000169154.jpg | .jpg | True
000000170990.jpg | .jpg | True
000000220430.jpg | .jpg | True
000000078640.jpg | .jpg | True
000000401622.jpg | .jpg | True
000000063476.jpg | .jpg | True
000000436693.jpg | .jpg | True
000000232280.jpg | .jpg | True
000000514404.jpg | .jpg | True
000000393721.jpg | .jpg | True
000000222056.jpg | .jpg | True


In [11]:
from collections import Counter

extension_counts = Counter(
    file.suffix.lower()
    for file in TRAIN_IMAGES.iterdir()
    if file.is_file()
)

print(extension_counts)

Counter({'.jpg': 61705})


In [12]:
import json

with open(TRAIN_CAPTIONS, "r") as f:
    train_data = json.load(f)

print("First 10 filenames in COCO annotations:")

for image in train_data["images"][:10]:
    print(image["file_name"])

First 10 filenames in COCO annotations:
000000391895.jpg
000000522418.jpg
000000184613.jpg
000000318219.jpg
000000554625.jpg
000000574769.jpg
000000060623.jpg
000000309022.jpg
000000005802.jpg
000000222564.jpg


In [14]:
import json
import pandas as pd
import random

with open(TRAIN_CAPTIONS, "r") as f:
    train_data = json.load(f)

with open(VAL_CAPTIONS, "r") as f:
    val_data = json.load(f)

print("Train annotation images:", len(train_data["images"]))
print("Validation annotation images:", len(val_data["images"]))

Train annotation images: 118287
Validation annotation images: 5000


In [15]:
SEED = 42

train_image_records = train_data["images"]

available_train_records = [
    image for image in train_image_records
    if (TRAIN_IMAGES / image["file_name"]).exists()
]

print("Available training images:", len(available_train_records))

assert len(available_train_records) >= 10_000

rng = random.Random(SEED)

knowledge_base_records = rng.sample(
    available_train_records,
    10_000
)

print("Selected knowledge-base images:", len(knowledge_base_records))

Available training images: 61705
Selected knowledge-base images: 10000


In [16]:
val_image_records = val_data["images"]

available_val_records = [
    image for image in val_image_records
    if (VAL_IMAGES / image["file_name"]).exists()
]

assert len(available_val_records) >= 200

val_rng = random.Random(SEED)

selected_val_records = val_rng.sample(
    available_val_records,
    200
)

validation_records = selected_val_records[:100]
testing_records = selected_val_records[100:]

print("Validation images:", len(validation_records))
print("Testing images:", len(testing_records))

Validation images: 100
Testing images: 100


In [17]:
def create_manifest(records, image_directory, split_name):
    return pd.DataFrame([
        {
            "image_id": int(record["id"]),
            "file_name": record["file_name"],
            "image_path": str(image_directory / record["file_name"]),
            "split": split_name
        }
        for record in records
    ])

knowledge_base_manifest = create_manifest(
    knowledge_base_records,
    TRAIN_IMAGES,
    "knowledge_base"
)

validation_manifest = create_manifest(
    validation_records,
    VAL_IMAGES,
    "validation"
)

testing_manifest = create_manifest(
    testing_records,
    VAL_IMAGES,
    "testing"
)

In [18]:
kb_ids = set(knowledge_base_manifest["image_id"])
val_ids = set(validation_manifest["image_id"])
test_ids = set(testing_manifest["image_id"])

assert len(kb_ids) == 10_000
assert len(val_ids) == 100
assert len(test_ids) == 100

assert kb_ids.isdisjoint(val_ids)
assert kb_ids.isdisjoint(test_ids)
assert val_ids.isdisjoint(test_ids)

assert knowledge_base_manifest["image_path"].map(
    lambda p: Path(p).exists()
).all()

assert validation_manifest["image_path"].map(
    lambda p: Path(p).exists()
).all()

assert testing_manifest["image_path"].map(
    lambda p: Path(p).exists()
).all()

print("All split checks passed.")

All split checks passed.


In [19]:
SPLIT_DIR = PROJECT_ROOT / "reports" / "splits"
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

knowledge_base_manifest.to_parquet(
    SPLIT_DIR / "knowledge_base_manifest.parquet",
    index=False
)

validation_manifest.to_parquet(
    SPLIT_DIR / "validation_manifest.parquet",
    index=False
)

testing_manifest.to_parquet(
    SPLIT_DIR / "testing_manifest.parquet",
    index=False
)

print("Final split manifests saved.")

Final split manifests saved.


In [20]:
train_captions_df = pd.DataFrame(train_data["annotations"])

kb_captions = train_captions_df[
    train_captions_df["image_id"].isin(kb_ids)
].copy()

print("Knowledge-base captions:", len(kb_captions))
print("Images represented:", kb_captions["image_id"].nunique())

Knowledge-base captions: 50018
Images represented: 10000


In [21]:
captions_per_image = kb_captions.groupby("image_id").size()

print(captions_per_image.describe())
print(captions_per_image.value_counts().sort_index())

count    10000.00000
mean         5.00180
std          0.04239
min          5.00000
25%          5.00000
50%          5.00000
75%          5.00000
max          6.00000
dtype: float64
5    9982
6      18
Name: count, dtype: int64


In [22]:
kb_captions.to_parquet(
    SPLIT_DIR / "knowledge_base_captions.parquet",
    index=False
)